# Conversor de moneda paso a paso

En este notebook explicamos el contenido de `conversorMoneda.py`: solicitamos dos países y una cantidad, buscamos la moneda de cada país y pedimos a REST Countries la conversión.

También veremos cómo solicitar una API key, guardarla en `.env` y evitar que Git la incluya en el repositorio.

## 1. Cómo funciona el conversor

El programa pregunta por un **país de origen**, un **país de destino** y la cantidad que se quiere convertir. A partir de cada país obtiene el código de su moneda y después solicita la conversión.

Por eso se escriben países como `Estados Unidos` y `España`, no monedas como `USD` o `EUR`. Una moneda puede ser compartida por varios países.

## 2. Instalar las dependencias

El programa utiliza `requests` para comunicarse con la API y `python-dotenv` para leer el archivo `.env`. Desde la carpeta raíz del proyecto, se pueden instalar con:

```powershell
python -m pip install -r .\Tema4-FuentesDatos\ejercicios_mios\requirements.txt
```

El fichero `requirements.txt` contiene las dependencias de este ejercicio.

## 3. Solicitar y obtener una API key

La API requiere una clave para autenticar las peticiones. Para obtener una:

1. Abre la página oficial de [registro de REST Countries](https://restcountries.com/sign-up).
2. Crea una cuenta gratuita con tus datos y acepta las condiciones del servicio.
3. Inicia sesión y sigue las indicaciones del proveedor para consultar o crear tu API key.
4. Copia la clave para guardarla localmente en el siguiente paso.

No compartas la clave ni la escribas en el código del notebook. La clave de demostración del proveedor no sirve para obtener una conversión real.

## 4. Localizar la carpeta del conversor

En un notebook no se puede usar `__file__` como en un script. Buscamos la carpeta que contiene `conversorMoneda.py` desde las ubicaciones habituales del kernel.

In [10]:
from pathlib import Path

posibles_carpetas = (
    Path.cwd(),
    Path.cwd() / 'Tema4-FuentesDatos' / 'ejercicios_mios',
    Path.cwd() / 'ejercicios_mios',
)
CARPETA = next(
    (
        ruta.resolve()
        for ruta in posibles_carpetas
        if (ruta / 'conversorMoneda.py').is_file()
    ),
    None,
)

if CARPETA is None:
    raise FileNotFoundError(
        'No se encontró la carpeta que contiene conversorMoneda.py.'
    )

print('Carpeta del conversor localizada.')

Carpeta del conversor localizada.


## 4.1. Instalar las dependencias en el kernel

El kernel activo de VS Code puede ser distinto del entorno `.venv` de la terminal. Ejecuta esta celda para instalar `requests` y `python-dotenv` en el mismo Python que está ejecutando el notebook. Hazlo antes de importar `dotenv` o guardar la clave.

In [11]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-r',
    str(CARPETA / 'requirements.txt'),
])
print('Dependencias instaladas en el kernel del notebook.')

Dependencias instaladas en el kernel del notebook.


## 5. Guardar la clave en `.env`

El archivo `.env` guarda la clave fuera del código fuente. Para evitar que la clave quede guardada en el historial o en las salidas del notebook, no la solicitamos con `input()`. La celda siguiente crea una plantilla si el archivo todavía no existe.

Después, abre `.env` en el explorador de VS Code, sustituye el texto de ejemplo por tu clave y guarda el archivo. No ejecutes ni compartas una celda que contenga la clave.

```text
RESTCOUNTRIES_API_KEY=TU_CLAVE_API
```

La celda no sobrescribe un `.env` que ya exista.

In [12]:
ruta_env = CARPETA / '.env'
if not ruta_env.exists():
    ruta_env.write_text(
        'RESTCOUNTRIES_API_KEY=TU_CLAVE_API\n',
        encoding='utf-8',
    )
    print('Archivo .env creado. Ábrelo y sustituye TU_CLAVE_API por tu clave.')
else:
    print('El archivo .env ya existe. Ábrelo para comprobar o actualizar la clave.')

El archivo .env ya existe. Ábrelo para comprobar o actualizar la clave.


## 6. Evitar que Git suba `.env`

`.gitignore` indica a Git qué ficheros locales debe excluir. La siguiente celda añade `.env` si todavía no aparece en el `.gitignore` de esta carpeta, conservando el resto de sus reglas. Así evitamos incluir la clave al preparar cambios para Git.

In [13]:
ruta_gitignore = CARPETA / '.gitignore'
contenido_gitignore = (
    ruta_gitignore.read_text(encoding='utf-8')
    if ruta_gitignore.exists()
    else ''
)
reglas = {linea.strip() for linea in contenido_gitignore.splitlines()}

if '.env' not in reglas:
    if contenido_gitignore and not contenido_gitignore.endswith(('\n', '\r')):
        contenido_gitignore += '\n'
    ruta_gitignore.write_text(
        contenido_gitignore + '.env\n',
        encoding='utf-8',
    )

print('.env está incluido en el .gitignore local.')

.env está incluido en el .gitignore local.


Comprueba la regla desde PowerShell, en la carpeta del ejercicio:

```powershell
git check-ignore -v .env
```

El comando debe mostrar la regla que está ignorando `.env`. No basta con ignorar el fichero si ya se había añadido a Git anteriormente; en ese caso hay que quitarlo del seguimiento sin borrar la copia local.

## 7. Importar módulos y configurar la API

`os` lee variables de entorno, `unicodedata` ayuda a comparar nombres con o sin tildes y `Decimal` permite manejar cantidades decimales. `requests` realiza las peticiones HTTP y `load_dotenv` lee la clave guardada en `.env`.

In [14]:
import os
import unicodedata
from decimal import Decimal, InvalidOperation
from pathlib import Path

import requests
from dotenv import load_dotenv

In [15]:
BASE_URL = 'https://api.restcountries.com'
URL_PAIS = f'{BASE_URL}/countries/v5'
URL_CONVERSION = f'{BASE_URL}/currencies/v1/convert'
TIMEOUT = 20
RESPONSE_FIELDS = (
    'names.common,names.official,names.alternates,names.native,'
    'names.translations,currencies,codes.alpha_3'
)

## 8. Preparar y comparar nombres de países

`normalizar` convierte el texto a minúsculas y quita las marcas de acentuación para que las comparaciones no dependan de mayúsculas o tildes. `recorrer_textos` busca nombres dentro de estructuras anidadas como traducciones y nombres nativos.

In [16]:
def normalizar(texto):
    '''Compara nombres ignorando mayúsculas y tildes.'''
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    ).casefold()


def recorrer_textos(valor):
    '''Devuelve los textos encontrados dentro de nombres anidados.'''
    if isinstance(valor, str):
        yield valor
    elif isinstance(valor, dict):
        for elemento in valor.values():
            yield from recorrer_textos(elemento)
    elif isinstance(valor, list):
        for elemento in valor:
            yield from recorrer_textos(elemento)

## 9. Centralizar las peticiones HTTP

`pedir_datos` envía una petición con `requests`, comprueba los errores HTTP y valida el formato JSON. También detecta las respuestas de demostración, que no contienen resultados reales de conversión.

In [17]:
def pedir_datos(session, url, params=None):
    respuesta = session.get(url, params=params, timeout=TIMEOUT)
    respuesta.raise_for_status()
    contenido = respuesta.json()

    if contenido.get('errors'):
        mensajes = '; '.join(
            error.get('message', 'Error de la API')
            for error in contenido['errors']
        )
        raise RuntimeError(mensajes)

    datos = contenido.get('data')
    if not isinstance(datos, dict):
        raise RuntimeError('La respuesta de la API no tiene el formato esperado.')
    if '_demo' in datos:
        raise RuntimeError(
            'La API devolvió datos de demostración, no resultados reales. '
            'Configura una clave API propia en RESTCOUNTRIES_API_KEY.'
        )

    return datos

## 10. Buscar un país

La búsqueda prueba el nombre común y las traducciones del país. Después comprueba coincidencias exactas y evita aceptar una búsqueda ambigua. El resultado contiene, entre otros campos, los nombres, el código ISO y las monedas.

In [18]:
def buscar_pais(session, nombre_pais):
    '''Busca un país por su nombre común, nativo o traducido.'''
    parametros = {
        'q': nombre_pais,
        'limit': 100,
        'response_fields': RESPONSE_FIELDS,
    }

    for endpoint in ('name', 'names.translations'):
        datos = pedir_datos(
            session,
            f'{URL_PAIS}/{endpoint}',
            params=parametros,
        )
        paises = datos.get('objects', [])
        nombre_buscado = normalizar(nombre_pais)
        coincidencias = []

        for pais in paises:
            nombres = pais.get('names', {})
            nombres_disponibles = {
                normalizar(texto)
                for texto in recorrer_textos(nombres)
            }
            if nombre_buscado in nombres_disponibles:
                coincidencias.append(pais)

        paises_unicos = {}
        for pais in coincidencias:
            codigo = pais.get('codes', {}).get('alpha_3')
            if codigo:
                paises_unicos[codigo] = pais

        if len(paises_unicos) == 1:
            return next(iter(paises_unicos.values()))
        if len(paises_unicos) > 1:
            raise ValueError(
                f'El nombre «{nombre_pais}» coincide con varios países. '
                'Prueba con el nombre oficial o en inglés.'
            )

    raise ValueError(
        f'No se ha encontrado el país «{nombre_pais}». '
        'Prueba con su nombre común, nativo o traducido.'
    )

## 11. Obtener la moneda del país

Si el país tiene una sola moneda, se selecciona automáticamente. Si tiene varias, el programa muestra sus códigos y pide elegir uno.

In [19]:
def seleccionar_moneda(pais):
    '''Obtiene el código de moneda del país; pregunta si hay varios.'''
    monedas = pais.get('currencies', [])

    if isinstance(monedas, dict):
        monedas = [
            {'code': codigo, **datos}
            for codigo, datos in monedas.items()
            if isinstance(datos, dict)
        ]

    monedas = [
        moneda for moneda in monedas
        if isinstance(moneda, dict) and moneda.get('code')
    ]
    if not monedas:
        raise ValueError(
            f"No se encontró una moneda para {pais['names']['common']}."
        )
    if len(monedas) == 1:
        return monedas[0]

    print(f"{pais['names']['common']} tiene varias monedas:")
    for moneda in monedas:
        print(f"  {moneda['code']}: {moneda.get('name', 'Sin nombre')}")

    codigos = {moneda['code'].upper(): moneda for moneda in monedas}
    while True:
        codigo = input('Escribe el código de la moneda que quieres usar: ')
        moneda = codigos.get(codigo.strip().upper())
        if moneda:
            return moneda
        print('Código no válido; elige uno de la lista.')

## 12. Validar la cantidad

`Decimal` evita algunos errores de precisión habituales al trabajar con números decimales. Se aceptan cantidades escritas con punto o coma, pero deben ser finitas y mayores que cero.

In [20]:
def pedir_cantidad():
    while True:
        texto = input('Cantidad a convertir (por ejemplo, 100): ').strip()
        try:
            cantidad = Decimal(texto.replace(',', '.'))
        except InvalidOperation:
            print('Escribe una cantidad numérica válida.')
            continue

        if cantidad.is_finite() and cantidad > 0:
            return cantidad
        print('La cantidad debe ser un número mayor que cero.')

## 13. Solicitar la conversión

La petición envía los códigos de moneda de origen y destino, además de la cantidad. La respuesta incluye el resultado y la tasa aplicada.

In [21]:
def convertir(session, moneda_origen, moneda_destino, cantidad):
    parametros = {
        'from': moneda_origen['code'],
        'to': moneda_destino['code'],
        'amount': str(cantidad),
    }
    datos = pedir_datos(session, URL_CONVERSION, params=parametros)
    resultados = datos.get('objects', [])
    if not resultados:
        raise RuntimeError('La API no devolvió el resultado de la conversión.')

    resultado = resultados[0]
    cantidad_convertida = Decimal(str(resultado['result']))
    tasa = Decimal(str(resultado['rate']))
    return cantidad_convertida, tasa

## 14. Coordinar el programa

`main` carga `.env`, crea una sesión HTTP con la cabecera `Authorization: Bearer ...`, solicita los datos al usuario y coordina las búsquedas y la conversión. La clave se lee desde el entorno y nunca se imprime.

In [22]:
def main():
    load_dotenv(dotenv_path=CARPETA / '.env')
    api_key = os.getenv('RESTCOUNTRIES_API_KEY')
    if not api_key:
        raise RuntimeError(
            'Falta la clave API. Añádela como RESTCOUNTRIES_API_KEY '
            'en el archivo .env.'
        )

    pais_origen = input('País de origen: ').strip()
    pais_destino = input('País de destino: ').strip()
    cantidad = pedir_cantidad()

    session = requests.Session()
    session.headers.update({
        'Authorization': f'Bearer {api_key}',
        'Accept': 'application/json',
    })

    datos_origen = buscar_pais(session, pais_origen)
    datos_destino = buscar_pais(session, pais_destino)
    moneda_origen = seleccionar_moneda(datos_origen)
    moneda_destino = seleccionar_moneda(datos_destino)

    cantidad_convertida, tasa = convertir(
        session,
        moneda_origen,
        moneda_destino,
        cantidad,
    )

    print(
        f"{cantidad} {moneda_origen['code']} "
        f"({moneda_origen.get('name', '')}) = "
        f"{cantidad_convertida:.2f} {moneda_destino['code']} "
        f"({moneda_destino.get('name', '')})"
    )
    print(
        f"Tasa aplicada: 1 {moneda_origen['code']} = "
        f"{tasa} {moneda_destino['code']}"
    )

## 15. Ejecutar el conversor

Ejecuta la última celda cuando tengas una API key propia guardada en `.env`. Esta celda inicia una conversación con el programa y realiza peticiones reales a REST Countries. Por ejemplo, se pueden introducir `Estados Unidos`, `España` y `100`.

In [23]:
try:
    main()
except (requests.RequestException, RuntimeError, ValueError) as error:
    print(f'No se pudo completar la conversión: {error}')

400 BRL (Brazilian real) = 64.06 EUR (Euro)
Tasa aplicada: 1 BRL = 0.16014286671487576 EUR
